In [ ]:
# Project FORESIGHT — 03: Baseline Demand Forecasting

### Objective
This notebook establishes baseline demand forecasting models and metric evaluation pipelines for **Project FORESIGHT**.

In accordance with the project architecture guidelines:
1. **Data Loading**: Import weekly aggregated sales and calendar data from `data/`.
2. **Baseline Construction**: Build a 52-week Seasonal-Naive model with a 1-week fallback mechanism.
3. **Metric Calculation**: Implement and evaluate Weighted Absolute Percentage Error ($WAPE$).
4. **Baseline Performance**: Establish ground-truth error metrics to compare against machine learning models.


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Display configurations
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Environment setup complete.")

Environment setup complete.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Load processed datasets generated by Member 1
df_sales = pd.read_csv('processed_sales.csv')
df_cal = pd.read_csv('processed_calendar.csv')

df_sales['Date'] = pd.to_datetime(df_sales['Date'])
df_cal['date'] = pd.to_datetime(df_cal['date'])

print("Sales records loaded:", len(df_sales))
print("Calendar records loaded:", len(df_cal))

Sales records loaded: 36550
Calendar records loaded: 731


In [3]:
# 1. Aggregate Sales to Weekly Level
weekly_sales = (
    df_sales.groupby(['SKU', pd.Grouper(key='Date', freq='W-MON')])
    .agg(
        Units_Sold=('Units_Sold', 'sum'),
        Price=('Price', 'mean'),
        Promotion=('Promotion', 'max')
    )
    .reset_index()
    .rename(columns={'Date': 'Week_Start'})
    .sort_values(['SKU', 'Week_Start'])
)

In [4]:
# 2. Build 52-Week Lag Prediction (Seasonal-Naive)
weekly_sales['lag_52'] = weekly_sales.groupby('SKU')['Units_Sold'].shift(52)
weekly_sales['lag_1'] = weekly_sales.groupby('SKU')['Units_Sold'].shift(1)
weekly_sales['baseline_pred'] = weekly_sales['lag_52'].fillna(weekly_sales['lag_1']).fillna(0)

In [5]:
# 3. WAPE Metric Calculation
def calculate_wape(y_true, y_pred):
    total_actual = np.sum(np.abs(y_true))
    if total_actual == 0:
        return 0.0
    return np.sum(np.abs(y_true - y_pred)) / total_actual

valid_mask = weekly_sales['lag_1'].notna()
baseline_wape = calculate_wape(
    weekly_sales.loc[valid_mask, 'Units_Sold'],
    weekly_sales.loc[valid_mask, 'baseline_pred']
)

print(f"Overall Seasonal-Naive Baseline WAPE: {baseline_wape:.4f}")

Overall Seasonal-Naive Baseline WAPE: 0.1281
